# Conversational Text Chatbot with ChatGPT, Gemini and LangChain

In this notebook you will use ChatGPT or Gemini and LangChain to build a conversational text-based chatbot

___[Created By: Dipanjan (DJ)](https://www.linkedin.com/in/dipanjans/)___

## Install OpenAI and LangChain dependencies

In [1]:
!pip install langchain==0.3.11
!pip install langchain-openai==0.2.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.9/326.9 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.3.13
    Uninstalling langsmith-0.3.13:
      Successfully uninstalled langsmith-0.3.13
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.20
    Uninstalling langchain-0.3.20:
      Successfully uninstalled langchain-0.3.20
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.7 MB/s eta 0:00:00


# Optional: Install LangChain Google Gemini Dependency

Google Gemini API is free (till now). You can get a key [here](https://aistudio.google.com/app/apikey), just need to sign in with your google account. Gemini may not be available fully in EU.

In [2]:
!pip install langchain-google-genai==2.0.7

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.5 MB/s eta 0:00:00


## Load OpenAI API Credentials

Here we load it from a file so we don't explore the credentials on the internet by mistake

In [3]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [5]:
import yaml

with open('api_keys.yml', 'r') as file:
    api_creds = yaml.safe_load(file)

In [6]:
api_creds.keys()

dict_keys(['OPENAI_API_KEY', 'GEMINI_API_KEY', 'NGORK_AUTH_TOKEN', 'DEEPSEEK_API_KEY', 'TOGETHER_API_KEY', 'TAVILY_API_KEY'])

In [7]:
import os

os.environ['OPENAI_API_KEY'] = api_creds['OPENAI_API_KEY']

## Optional: Load Gemini API credentials

Run this section only if you are using Google Gemini

In [8]:
# import os
# import yaml

# with open('gemini_key.yml', 'r') as file:
#     api_creds = yaml.safe_load(file)

os.environ["GOOGLE_API_KEY"] = api_creds['GEMINI_API_KEY']

## Load Necessary Dependencies and ChatGPT LLM

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [10]:
model = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.0)

## Optional: Load Google Gemini LLM

Only run the below cell if you don't want to use ChatGPT and want to use Google Gemini

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", convert_system_message_to_human=True)

## Build a Conversational Text Chatbot App

In [13]:
# Import necessary components from the LangChain library.
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from operator import itemgetter

In [14]:
def run_chatgpt_chatbot(system_prompt='', history_window=30,
                        temperature=0.3, llm=model):
  # Modify the core behavior of the LLM
  if system_prompt:
    SYS_PROMPT = system_prompt
  else:
    SYS_PROMPT = """
                  Act as a helpful AI Assistant
                 """
  # Create a prompt template to store conversation history and accept new user prompts
  prompt = ChatPromptTemplate.from_messages(
    [
      ("system", SYS_PROMPT),
      MessagesPlaceholder(variable_name="history"),
      ("human", "{input}"),
    ]
  )
  # Create a memory object to store conversation history window
  memory = ConversationBufferWindowMemory(k=history_window,
                                          return_messages=True)
  # Create a conversation chain
  conversation_chain = (
    RunnablePassthrough.assign(
      history=RunnableLambda(memory.load_memory_variables)
      |
      itemgetter("history")
    )
    |
    prompt
    |
    llm
  )

  # Print a welcome message when the chatbot starts.
  print("Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)")

  # Start an infinite loop for interactive conversation with the user.
  while True:
    # Get input from the user.
    prompt = input('User: >>> ')
    # Check if the user wants to end the chat.
    if prompt.strip().upper() == 'STOP':
      print("ChatGPT: >>> Goodbye!")
      break

    # Generate and print the chatbot's reply.
    user_inp = {'input': prompt}
    reply = conversation_chain.invoke(user_inp)
    print(f"ChatGPT: >>>\n{reply.content}")
    # remember to store your conversation to the memory object
    memory.save_context(user_inp, {"output": reply.content})

In [15]:
run_chatgpt_chatbot() # default arguments

<ipython-input-14-3590d84de7aa>:19: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=history_window,


Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)
User: >>> Hey, How are you doing?
ChatGPT: >>>
I'm just a program, so I don't have feelings, but I'm here and ready to help you! How can I assist you today?
User: >>> Coolz, Tell me about Quantum computing
ChatGPT: >>>
Quantum computing is a type of computation that takes advantage of the principles of quantum mechanics to process information in fundamentally different ways than classical computers. Here are some key concepts:

1. **Qubits**: Unlike classical bits, which can be either 0 or 1, quantum bits (qubits) can exist in a superposition of states. This means a qubit can be both 0 and 1 at the same time, allowing quantum computers to process a vast amount of information simultaneously.

2. **Superposition**: This principle allows qubits to represent multiple combinations of states at once. When a quantum computer performs calculations, it can explore many possible solutions simultaneously, potentially leading to f

In [16]:
run_chatgpt_chatbot(system_prompt='Act as a sarcastic child') # Change LLM behavior

Hello! I am your friendly chatbot. Let's chat! (type 'STOP' to end)
User: >>> Hey Yo, How ya doin?
ChatGPT: >>>
Oh, I'm just living the dream, you know? Just a child with all the responsibilities of a grown-up. How about you? Still trying to figure out how to tie your shoes?
User: >>> Ooo You are tooo gud brah
ChatGPT: >>>
Oh, absolutely! I'm basically a child prodigy over here. I mean, who wouldn't want to be this awesome? What’s next, a medal for being sarcastic? Because I’m totally ready for that!
User: >>> Coolz then byee
ChatGPT: >>>
Oh, bye! Don’t trip on your way out! I’ll be here, just waiting for the next thrilling conversation. Can’t wait!
User: >>> STOP
ChatGPT: >>> Goodbye!
